# 02. Khám phá dữ liệu bị thiếu

Notebook này thống kê các cột bị thiếu dữ liệu trong bảng chuẩn hóa và xuất riêng danh sách các dòng thiếu `commit_date` để kiểm tra xem repo có bị xóa hoặc chuyển sang riêng tư hay không.

Import thư viện và tải dữ liệu đã chuẩn hóa.

In [1]:
import os
import pandas as pd

DATA_PATH = "../data/interim/latest_standardized_data.csv"
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

EXTRACT_DIR = "../data/interim/missing_cases_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

Loaded dataset: 35,278 rows x 12 columns


Thống kê số lượng và tỷ lệ phần trăm giá trị bị thiếu ở từng cột.

In [2]:
missing_stats = pd.DataFrame({
    "missing_count": df_raw.isnull().sum(),
    "missing_pct": (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
})
missing_stats = missing_stats[missing_stats["missing_count"] > 0].sort_values(
    by="missing_pct", ascending=False
)
missing_stats

,missing_count,missing_pct
change_commit_message,22287,63.18
change_commit_date,22282,63.16
change_commit_url,22282,63.16
commit_url,1642,4.65
commit_date,1642,4.65


Kiểm tra các dòng bị thiếu `commit_date`. Vì `commit_url` cũng bị trống khi truy vấn thất bại, ta tạo thêm cột `repo_url` từ tên repo để script kiểm tra link có thể ping thử các repo này.

In [3]:
df_missing_commit_date = df_raw[df_raw["commit_date"].isnull()].copy()
df_missing_commit_date["repo_url"] = "https://github.com/" + df_missing_commit_date["repo"].astype(str)

print(f"Rows missing commit_date: {len(df_missing_commit_date):,}")
preview_cols = [c for c in ["id", "repo", "repo_url", "file_url"] if c in df_missing_commit_date.columns]
display(df_missing_commit_date[preview_cols].head(5))

out_path_1 = os.path.join(EXTRACT_DIR, "missing_commit_date.csv")
df_missing_commit_date.to_csv(out_path_1, index=False)
print(f"Exported {len(df_missing_commit_date):,} rows to {out_path_1}")

Rows missing commit_date: 1,642


,id,repo,repo_url,file_url
81,js__ai_created__588714688__7961ed1d86c9__204::...,harshitethic/Openai-clone,https://github.com/harshitethic/Openai-clone,https://github.com/harshitethic/Openai-clone/b...
154,js__generated_by_chatgpt__696458575__078fdd3ab...,arctyx-0/Bae-is-very-stinky,https://github.com/arctyx-0/Bae-is-very-stinky,https://github.com/arctyx-0/Bae-is-very-stinky...
293,python__generated_by_copilot__698621599__8b0f6...,Kreyren/shellcheck-bug-arrays-in-posix,https://github.com/Kreyren/shellcheck-bug-arra...,https://github.com/Kreyren/shellcheck-bug-arra...
306,python__suggested_by_copilot__698621599__8b0f6...,Kreyren/shellcheck-bug-arrays-in-posix,https://github.com/Kreyren/shellcheck-bug-arra...,https://github.com/Kreyren/shellcheck-bug-arra...
331,python__created_using_chatgpt__606835477__4378...,xieliaing/Side-Projects,https://github.com/xieliaing/Side-Projects,https://github.com/xieliaing/Side-Projects/blo...


Exported 1,642 rows to ../data/interim/missing_cases_extracted/missing_commit_date.csv


Kiểm tra các dòng thiếu `change_commit_date`. Theo mô tả từ nguồn dữ liệu gốc, ô trống nghĩa là không thu thập được lần thay đổi đầu tiên chứ không phải code chưa từng bị sửa. Ta chỉ xem mẫu vài dòng mà không xuất file CSV lớn để tránh tốn dung lượng.

In [4]:
df_missing_change = df_raw[df_raw["change_commit_date"].isnull()]
print(f"Rows missing change_commit_date: {len(df_missing_change):,} ({len(df_missing_change) / len(df_raw):.1%})")

preview_cols_2 = [c for c in ["id", "language", "file", "commit_url"] if c in df_missing_change.columns]
df_missing_change[preview_cols_2].head(5)

Rows missing change_commit_date: 22,282 (63.2%)


,id,language,file,commit_url
0,python__written_by_gpt__433861003__e397ad21a6e...,python,2022/python/d1-gpt.py,https://github.com/flowerett/aoc/commit/52a0fb...
1,python__written_by_gpt__433861003__e397ad21a6e...,python,2022/python/d1-gpt.py,https://github.com/flowerett/aoc/commit/52a0fb...
2,python__written_by_ai__437874902__14f89ee997ec...,python,2017/puzzle2.py,https://github.com/tipa16384/adventofcode/comm...
3,python__written_by_ai__437874902__14f89ee997ec...,python,2017/puzzle2.py,https://github.com/tipa16384/adventofcode/comm...
4,python__written_by_ai__437874902__1d766ed4953e...,python,2017/puzzle1.py,https://github.com/tipa16384/adventofcode/comm...


Kiểm tra xem có dòng nào bị thiếu từ 3 trường thông tin trở lên hay không và lưu ra file riêng nếu có.

In [5]:
missing_per_row = df_raw.isnull().sum(axis=1)
df_heavy_missing = df_raw[missing_per_row >= 3].copy()
print(f"Rows missing 3 or more fields: {len(df_heavy_missing):,}")

if len(df_heavy_missing) > 0:
    display(df_heavy_missing.head(5))
    out_path_3 = os.path.join(EXTRACT_DIR, "heavy_missing_rows.csv")
    df_heavy_missing.to_csv(out_path_3, index=False)
    print(f"Saved to {out_path_3}")

Rows missing 3 or more fields: 22,282


,id,repo,language,file_url,commit_url,file,comment,code_block,commit_date,change_commit_url,change_commit_date,change_commit_message
0,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/blob/71574e59...,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,2022-12-03T08:41:16Z,NaN,NaN,NaN
1,python__written_by_gpt__433861003__e397ad21a6e...,flowerett/aoc,python,https://github.com/flowerett/aoc/blob/71574e59...,https://github.com/flowerett/aoc/commit/52a0fb...,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,# solution written by GPT chatbot\n# works and...,2022-12-03T08:41:16Z,NaN,NaN,NaN
2,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/blob...,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle2.py,# Written by AI\n,# Written by AI\n\ndef spreadsheet_checksum(li...,2022-12-04T05:58:44Z,NaN,NaN,NaN
3,python__written_by_ai__437874902__14f89ee997ec...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/blob...,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle2.py,# Written by AI\n,# Written by AI\n\ndef spreadsheet_checksum(li...,2022-12-04T05:58:44Z,NaN,NaN,NaN
4,python__written_by_ai__437874902__1d766ed4953e...,tipa16384/adventofcode,python,https://github.com/tipa16384/adventofcode/blob...,https://github.com/tipa16384/adventofcode/comm...,2017/puzzle1.py,# Written by AI\n,"# Written by AI\n\nwith open('puzzle1.txt', 'r...",2022-12-04T05:58:44Z,NaN,NaN,NaN


Saved to ../data/interim/missing_cases_extracted/heavy_missing_rows.csv
